# Fill NaNs of weather

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


In [30]:
df = pd.read_csv('../Data/silver/dengue_weather_nans.csv')
df.head()

,idx_city,week,count,Latitude,Longitude,tavg,tmin,tmax,prcp,wdir,wspd,pres,elevation
0,COLOMBIA_AMAZONAS_LETICIA,2023-01-02,7,-4.212921,-69.942596,25.228571,22.985714,29.171429,8.528571,56.000000,4.800000,1010.985714,81.0
1,COLOMBIA_ANTIOQUIA_APARTADO,2023-01-02,7,7.884901,-76.622746,27.628571,22.357143,30.885714,5.000000,180.714286,8.214286,1012.014286,29.0
2,COLOMBIA_ANTIOQUIA_CARACOLI,2023-01-02,1,6.409276,-74.756698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,616.0
3,COLOMBIA_ANTIOQUIA_CAREPA,2023-01-02,2,7.798452,-76.746039,27.628571,22.357143,30.885714,5.000000,180.714286,8.214286,1012.014286,10.0
4,COLOMBIA_ANTIOQUIA_CAUCASIA,2023-01-02,3,7.987758,-75.198374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.0


In [32]:
df.dropna(subset = ['elevation'], inplace = True)

In [33]:
data_full = df.dropna(subset = ['tavg']).copy()
data_na = df[df['tavg'].isna()].copy()

In [67]:
data_full.columns

Index(['idx_city', 'week', 'count', 'Latitude', 'Longitude', 'tavg', 'tmin',
       'tmax', 'prcp', 'wdir', 'wspd', 'pres', 'elevation'],
      dtype='object')

In [34]:
data_na

,idx_city,week,count,Latitude,Longitude,tavg,tmin,tmax,prcp,wdir,wspd,pres,elevation
2,COLOMBIA_ANTIOQUIA_CARACOLI,2023-01-02,1,6.409276,-74.756698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,616.0
4,COLOMBIA_ANTIOQUIA_CAUCASIA,2023-01-02,3,7.987758,-75.198374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.0
7,COLOMBIA_ANTIOQUIA_EL BAGRE,2023-01-02,3,7.697710,-74.622203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,518.0
10,COLOMBIA_ANTIOQUIA_MUTATA,2023-01-02,1,7.244068,-76.436542,NaN,NaN,NaN,NaN,NaN,NaN,NaN,133.0
11,COLOMBIA_ANTIOQUIA_NECHI,2023-01-02,5,8.093808,-74.775199,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15368,COLOMBIA_VALLE_SAN PEDRO,2023-12-25,4,3.995568,-76.228049,NaN,NaN,NaN,NaN,NaN,NaN,NaN,990.0
15369,COLOMBIA_VALLE_SEVILLA,2023-12-25,2,4.264507,-75.934396,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1622.0
15370,COLOMBIA_VALLE_TRUJILLO,2023-12-25,3,4.212304,-76.318893,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1313.0
15371,COLOMBIA_VALLE_TULUA,2023-12-25,34,4.085667,-76.197278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,976.0


In [35]:
dates = df['week'].unique()

In [36]:
# simplest example
df_1 = df[df['week'] == dates[0]].copy()
df_1_full = df_1.dropna(subset = ['tavg', 'elevation']).copy()
df_1_na = df_1[df_1['tavg'].isna()].copy()

In [37]:
y = df_1_full['tavg']
X = df_1_full[['Latitude', 'Longitude', 'elevation']]

In [39]:
X_na = df_1_na[['Latitude', 'Longitude', 'elevation']]

In [46]:
## random forest regression
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)
# predict y for the missing values
y_pred = model.predict(X)
r2 = model.score(X, y)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f'R2: {r2:.3f}, RMSE: {rmse:.3f}')
y_pred2 = model.predict(X_na)

R2: 0.962, RMSE: 0.492


In [47]:
df_1_na['tavg'] = y_pred2
df_1_na

,idx_city,week,count,Latitude,Longitude,tavg,tmin,tmax,prcp,wdir,wspd,pres,elevation
2,COLOMBIA_ANTIOQUIA_CARACOLI,2023-01-02,1,6.409276,-74.756698,23.717571,NaN,NaN,NaN,NaN,NaN,NaN,616.0
4,COLOMBIA_ANTIOQUIA_CAUCASIA,2023-01-02,3,7.987758,-75.198374,26.707143,NaN,NaN,NaN,NaN,NaN,NaN,59.0
7,COLOMBIA_ANTIOQUIA_EL BAGRE,2023-01-02,3,7.697710,-74.622203,24.373000,NaN,NaN,NaN,NaN,NaN,NaN,518.0
10,COLOMBIA_ANTIOQUIA_MUTATA,2023-01-02,1,7.244068,-76.436542,26.763857,NaN,NaN,NaN,NaN,NaN,NaN,133.0
11,COLOMBIA_ANTIOQUIA_NECHI,2023-01-02,5,8.093808,-74.775199,27.184143,NaN,NaN,NaN,NaN,NaN,NaN,31.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,COLOMBIA_TOLIMA_PURIFICACION,2023-01-02,1,3.858211,-74.930822,24.722714,NaN,NaN,NaN,NaN,NaN,NaN,311.0
227,COLOMBIA_VALLE_BUGA,2023-01-02,2,3.900058,-76.302013,22.442857,NaN,NaN,NaN,NaN,NaN,NaN,976.0
230,COLOMBIA_VALLE_DAGUA,2023-01-02,1,3.658161,-76.689397,22.462000,NaN,NaN,NaN,NaN,NaN,NaN,881.0
232,COLOMBIA_VALLE_EL DOVIO,2023-01-02,1,4.511899,-76.238102,21.131714,NaN,NaN,NaN,NaN,NaN,NaN,1430.0


# Loop 

In [63]:
df_no_data = data_na[['idx_city', 'week']].copy()

In [66]:
from tqdm import tqdm
df_filled = pd.DataFrame()
vars_to_fill = [  'tavg', 'tmin', 'tmax', 'prcp', 'wdir', 'wspd', 'pres']
dates = df['week'].unique()
predictors = ['Latitude', 'Longitude', 'elevation']

for date in tqdm(dates):
    df_date = df[df['week'] == date].copy()
    df_date_full = df_date.dropna(subset = vars_to_fill + ['elevation']).copy()
    df_date_na = df_date[df_date[vars_to_fill].isna().any(axis=1)].copy()
    for target in vars_to_fill:
        
        # partition data
        y = df_date_full[target]
        X = df_date_full[['Latitude', 'Longitude', 'elevation']]
        X_na = df_date_na[['Latitude', 'Longitude', 'elevation']]
        ## random forest regression
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X, y)
        y_pred_na = model.predict(X_na)
        df_date_na[target] = y_pred_na
    df_filled = pd.concat([df_filled, df_date_na], axis=0)

df_filled

100%|██████████| 52/52 [01:06<00:00,  1.27s/it]


,idx_city,week,count,Latitude,Longitude,tavg,tmin,tmax,prcp,wdir,wspd,pres,elevation
2,COLOMBIA_ANTIOQUIA_CARACOLI,2023-01-02,1,6.409276,-74.756698,23.717571,19.421714,28.475857,5.852143,199.725714,5.710857,1013.233143,616.0
4,COLOMBIA_ANTIOQUIA_CAUCASIA,2023-01-02,3,7.987758,-75.198374,26.707143,22.419857,30.750000,3.897429,181.694286,7.572714,1011.725143,59.0
7,COLOMBIA_ANTIOQUIA_EL BAGRE,2023-01-02,3,7.697710,-74.622203,24.373000,19.955000,28.775429,4.513857,188.971429,5.061857,1013.150857,518.0
10,COLOMBIA_ANTIOQUIA_MUTATA,2023-01-02,1,7.244068,-76.436542,26.763857,22.218286,31.040714,4.482429,186.354286,6.674143,1012.181714,133.0
11,COLOMBIA_ANTIOQUIA_NECHI,2023-01-02,5,8.093808,-74.775199,27.184143,22.520571,31.021714,3.865571,190.724286,6.566143,1011.844429,31.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15368,COLOMBIA_VALLE_SAN PEDRO,2023-12-25,4,3.995568,-76.228049,23.540000,19.688857,29.701000,10.327571,80.534286,8.008000,1017.057143,990.0
15369,COLOMBIA_VALLE_SEVILLA,2023-12-25,2,4.264507,-75.934396,22.341429,17.999143,29.141714,11.801571,161.238571,5.438429,1017.891857,1622.0
15370,COLOMBIA_VALLE_TRUJILLO,2023-12-25,3,4.212304,-76.318893,23.419143,19.692714,29.193857,10.961571,98.148571,7.315571,1017.131429,1313.0
15371,COLOMBIA_VALLE_TULUA,2023-12-25,34,4.085667,-76.197278,23.611714,19.714429,29.531429,10.350143,89.347143,7.991286,1016.966857,976.0


In [69]:
data_full.columns

Index(['idx_city', 'week', 'count', 'Latitude', 'Longitude', 'tavg', 'tmin',
       'tmax', 'prcp', 'wdir', 'wspd', 'pres', 'elevation'],
      dtype='object')

In [70]:
df_filled = df_filled[['idx_city', 'week', 'count', 'Latitude', 'Longitude', 'tavg', 'tmin',
       'tmax', 'prcp', 'wdir', 'wspd', 'pres', 'elevation']]

In [71]:
df_final = pd.concat([data_full, df_filled], axis=0)

In [72]:
df_final

,idx_city,week,count,Latitude,Longitude,tavg,tmin,tmax,prcp,wdir,wspd,pres,elevation
0,COLOMBIA_AMAZONAS_LETICIA,2023-01-02,7,-4.212921,-69.942596,25.228571,22.985714,29.171429,8.528571,56.000000,4.800000,1010.985714,81.0
1,COLOMBIA_ANTIOQUIA_APARTADO,2023-01-02,7,7.884901,-76.622746,27.628571,22.357143,30.885714,5.000000,180.714286,8.214286,1012.014286,29.0
3,COLOMBIA_ANTIOQUIA_CAREPA,2023-01-02,2,7.798452,-76.746039,27.628571,22.357143,30.885714,5.000000,180.714286,8.214286,1012.014286,10.0
5,COLOMBIA_ANTIOQUIA_CHIGORODO,2023-01-02,2,7.612938,-76.638426,27.628571,22.357143,30.885714,5.000000,180.714286,8.214286,1012.014286,46.0
6,COLOMBIA_ANTIOQUIA_COPACABANA,2023-01-02,2,6.346105,-75.508026,21.671429,15.828571,25.857143,6.071429,142.000000,7.857143,1018.957143,1421.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15368,COLOMBIA_VALLE_SAN PEDRO,2023-12-25,4,3.995568,-76.228049,23.540000,19.688857,29.701000,10.327571,80.534286,8.008000,1017.057143,990.0
15369,COLOMBIA_VALLE_SEVILLA,2023-12-25,2,4.264507,-75.934396,22.341429,17.999143,29.141714,11.801571,161.238571,5.438429,1017.891857,1622.0
15370,COLOMBIA_VALLE_TRUJILLO,2023-12-25,3,4.212304,-76.318893,23.419143,19.692714,29.193857,10.961571,98.148571,7.315571,1017.131429,1313.0
15371,COLOMBIA_VALLE_TULUA,2023-12-25,34,4.085667,-76.197278,23.611714,19.714429,29.531429,10.350143,89.347143,7.991286,1016.966857,976.0


In [79]:
df_final.sort_values(by=['week', 'idx_city'], inplace=True)
df_final.reset_index(drop=True, inplace=True)
df_final

,idx_city,week,count,Latitude,Longitude,tavg,tmin,tmax,prcp,wdir,wspd,pres,elevation
0,COLOMBIA_AMAZONAS_LETICIA,2023-01-02,7,-4.212921,-69.942596,25.228571,22.985714,29.171429,8.528571,56.000000,4.800000,1010.985714,81.0
1,COLOMBIA_ANTIOQUIA_APARTADO,2023-01-02,7,7.884901,-76.622746,27.628571,22.357143,30.885714,5.000000,180.714286,8.214286,1012.014286,29.0
2,COLOMBIA_ANTIOQUIA_CARACOLI,2023-01-02,1,6.409276,-74.756698,23.717571,19.421714,28.475857,5.852143,199.725714,5.710857,1013.233143,616.0
3,COLOMBIA_ANTIOQUIA_CAREPA,2023-01-02,2,7.798452,-76.746039,27.628571,22.357143,30.885714,5.000000,180.714286,8.214286,1012.014286,10.0
4,COLOMBIA_ANTIOQUIA_CAUCASIA,2023-01-02,3,7.987758,-75.198374,26.707143,22.419857,30.750000,3.897429,181.694286,7.572714,1011.725143,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14935,COLOMBIA_VALLE_VIJES,2023-12-25,2,3.700400,-76.442888,23.514286,19.657143,29.728571,9.957143,53.142857,8.314286,1017.057143,974.0
14936,COLOMBIA_VALLE_YUMBO,2023-12-25,66,3.583466,-76.495222,23.514286,19.657143,29.728571,9.957143,53.142857,8.314286,1017.057143,999.0
14937,COLOMBIA_VALLE_ZARZAL,2023-12-25,7,4.393943,-76.070647,23.104571,19.476000,29.433571,9.983571,117.881429,6.187857,1015.992714,920.0
14938,COLOMBIA_VAUPES_MITU,2023-12-25,8,1.253850,-70.234558,25.814286,22.842857,31.428571,10.571429,173.428571,4.642857,1011.928571,175.0


In [80]:
df_final.to_csv('../Data/platinum/dengue_weather_interpolated.csv', index=False)